In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. Data Loading and Preprocessing
def load_and_preprocess_data(filename):
    df = pd.read_csv(filename)
    df['date'] = pd.to_datetime(df['date_id'], unit='D')
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    return df



In [3]:
def engineer_features(df):
    grouped = df.groupby(['symbol_id', 'date_id'])
    
    features = grouped.agg({
        'cummulative_continous_volume': ['max', 'mean', 'std'],
        'close_volume': 'max',
        'day_of_week': 'first',
        'month': 'first'
    }).reset_index()
    
    features.columns = ['symbol_id', 'date_id', 'max_volume', 'mean_volume', 'std_volume', 'close_volume', 'day_of_week', 'month']
    
    # Calculate volume change rate with error handling
    features['volume_change_rate'] = features.groupby('symbol_id')['max_volume'].pct_change()
    
    # Calculate moving averages
    features['ma_5'] = features.groupby('symbol_id')['close_volume'].rolling(window=5).mean().reset_index(0, drop=True)
    features['ma_10'] = features.groupby('symbol_id')['close_volume'].rolling(window=10).mean().reset_index(0, drop=True)
    
    # Handle infinite values
    features = features.replace([np.inf, -np.inf], np.nan)
    
    # Fill NaN values with median of the column
    for col in features.columns:
        if features[col].dtype in ['float64', 'int64']:
            features[col] = features[col].fillna(features[col].median())
    
    return features

In [4]:
def train_model(features):
    X = features.drop(['symbol_id', 'date_id', 'close_volume'], axis=1)
    y = features['close_volume']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Use RobustScaler instead of StandardScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Mean Squared Error: {mse}")
    
    return model, scaler

In [5]:
# 4. Prediction and Output Generation
def predict_close_volume(filename, model, scaler):
    # Load and preprocess the input data
    df = load_and_preprocess_data(filename)
    
    # Get the last date_id
    last_date_id = df['date_id'].max()
    
    # Filter data for the last date
    last_day_data = df[df['date_id'] == last_date_id]
    
    # Engineer features for the last day
    features = engineer_features(df)
    last_day_features = features[features['date_id'] == last_date_id]
    
    # Prepare the input for prediction
    X_pred = last_day_features.drop(['symbol_id', 'date_id', 'close_volume'], axis=1)
    X_pred_scaled = scaler.transform(X_pred)
    
    # Make predictions
    predictions = model.predict(X_pred_scaled)
    
    # Create output dataframe
    output = pd.DataFrame({
        'symbol_id': last_day_features['symbol_id'],
        'close_volume': predictions
    })
    
    return output

In [6]:
if __name__ == "__main__":
    # Load and preprocess training data
    train_df = load_and_preprocess_data("training.csv")
    
    # Engineer features
    features = engineer_features(train_df)
    
    # Train the model
    model, scaler = train_model(features)
    
    # Get input filename
    input_filename = 'sample1.csv'
    # Make predictions
    output = predict_close_volume(input_filename, model, scaler)
    
    # Save predictions to CSV
    output.to_csv("predictions.csv", index=False)
            
    print(output.to_string(index=False))

Mean Squared Error: 13338212977202.86
 symbol_id  close_volume
1029677210    2127795.85
1077784782    1826706.27
1084320043    1962412.78
1106122882    2183114.16
